# 检查 SToFM 官方 demo
本 Notebook 只读取本地已下载的 demo，不会自动下载语料或权重。没有 demo 时会输出明确提示。

In [ ]:
from pathlib import Path
import numpy as np
try:
    import anndata as ad
except ImportError:
    ad = None
root = Path('../data/raw/demo')
files = sorted(root.glob('*.h5ad')) if root.exists() else []
print('demo_h5ad_files=', [str(p) for p in files])

In [ ]:
if not files:
    print('未找到官方demo。请先查看 checkpoints/README.md，并用 scripts/download_demo.py 登记的安全方式下载。')
else:
    adata = ad.read_h5ad(files[0], backed='r')
    spatial_keys = [k for k in adata.obsm.keys() if 'spatial' in k.lower()]
    print('shape=', adata.shape)
    print('obs=', list(adata.obs.columns))
    print('var=', list(adata.var.columns))
    print('obsm=', list(adata.obsm.keys()))
    print('spatial_keys=', spatial_keys)

In [ ]:
if files and spatial_keys:
    import matplotlib.pyplot as plt
    xy = np.asarray(adata.obsm[spatial_keys[0]])
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(xy[:, 0], xy[:, 1], s=1)
    ax.set_title(f'{files[0].name}: {adata.n_obs} cells/spots, {adata.n_vars} genes')
    ax.set_aspect('equal')
    plt.show()
elif files:
    print('文件中没有自动识别到空间坐标；请检查obs或其他obsm键。')

## 嵌入检查
只有准备好固定 commit 的官方代码、`data.h5ad`、`hf.dataset`、cell encoder 和 SE(2)-Transformer 权重后，才运行官方 `get_embeddings.py`。预期输出 `stofm_emb.npy` 的第一维等于细胞数，第二维为 256。本 Notebook 不在缺少权重时生成假嵌入。

In [ ]:
embeddings = sorted(root.glob('stofm_emb.npy')) if root.exists() else []
if embeddings:
    emb = np.load(embeddings[0], mmap_mode='r')
    print('embedding_shape=', emb.shape, 'dtype=', emb.dtype)
else:
    print('尚未生成官方模型嵌入。')